In [ ]:
import pandas as pd
import numpy as np
small_frame = pd.read_pickle('....../COPD/features/20240225_phonation_features_Disvoice.pkl')
small_frame

In [ ]:
small_frame = small_frame.dropna()
small_frame

### lstm


#### mfcc

In [ ]:
import pandas as pd
frame_mfcc = pd.read_pickle('....../COPD/features/20240225_mfcc_features.pkl')
frame_mfcc

In [ ]:
# Merge DataFrames on 'name' column
merged_df = pd.merge(frame_mfcc, small_frame[['name']], on='name', how='inner')



# Filter rows in A based on whether 'name' is present in B
frame_mfcc = frame_mfcc[frame_mfcc['name'].isin(merged_df['name'])]

In [ ]:
frame_mfcc = frame_mfcc.reset_index(drop=True)
frame_mfcc

In [ ]:
frame_mfcc['new_feature'] = frame_mfcc['feature'].apply(lambda x: x[:30])
frame_mfcc

In [ ]:
X = frame_mfcc['new_feature'].to_list()
y = frame_mfcc['label'].to_list()

import numpy as np

X = np.vstack(X)
X.shape

In [ ]:
import pandas as pd
from tqdm import tqdm
import os
import librosa
import numpy as np

from matplotlib import pyplot as plt
import seaborn as sns


In [ ]:
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import precision_score, roc_auc_score, f1_score, recall_score, confusion_matrix
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn import metrics
from tensorflow.keras.datasets import cifar10
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Flatten, Conv2D, MaxPooling2D, Embedding
from tensorflow.keras.losses import sparse_categorical_crossentropy
from tensorflow.keras.optimizers import Adam
from sklearn.model_selection import KFold
import numpy as np
from tensorflow.keras.models import save_model,load_model

from keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense
from keras.layers import *
from tensorflow.keras.optimizers import SGD

from sklearn.model_selection import KFold, StratifiedKFold
from sklearn.metrics import auc
from sklearn.metrics import confusion_matrix
from sklearn.metrics import roc_auc_score
from sklearn.metrics import roc_curve
import matplotlib.pylab as plt
from sklearn.preprocessing import LabelEncoder, StandardScaler

from scipy import interp

from sklearn.metrics import r2_score, roc_auc_score, roc_curve, classification_report


# Assume your DataFrame is df with columns 'new_feature' and 'label'
scaler = StandardScaler()

# Initialize StratifiedKFold
skf = StratifiedKFold(n_splits=10, shuffle=True, random_state=0)

acc_arr = np.empty((10, 1))
f1_arr = np.empty((10, 1))
precision_arr = np.empty((10, 1))
recall_arr = np.empty((10, 1))
auc_arr = np.empty((10, 1))

sen_arr = np.empty((10, 1))
spe_arr = np.empty((10, 1))

ppv_arr = np.empty((10, 1))
npv_arr = np.empty((10, 1))

recall_arr = np.empty((10, 1))
pre_arr = np.empty((10, 1))
balanced_accuracy_scores_arr = np.empty((10, 1))

cnf_arr= []
x = 0

# Loop through each fold
for train_index, test_index in skf.split(X, y):
    X_train, X_test = X[train_index], X[test_index]
    y_train, y_test = np.array(y)[train_index], np.array(y)[test_index]

    # Scale the features using StandardScaler
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)


    df_x_train = X_train_scaled.reshape(-1,  1, X_train_scaled.shape[-1])
    df_x_test = X_test_scaled.reshape(-1,  1, X_test_scaled.shape[-1])
    df_y_train = y_train.reshape(-1, 1)
    df_y_test = y_test.reshape(-1, 1)

    model = load_model('....../0810/3rd_paper_LSTM/domains_fusion/mfcc/{0}'.format(x))


    y_prob = model.predict(df_x_test).ravel()

    pred = model.predict(df_x_test)
    y_pred = [np.round(x) for x in pred]
    test_preds1 = [np.round(x) for x in y_pred]
    test_preds1 = np.array(test_preds1)
    y_pred = test_preds1.flatten('A')

    df_y_test = np.array(df_y_test)
    df_y_test = df_y_test.flatten('A')
    

    print('Accuracy Score: {:.4f}'.format(accuracy_score(df_y_test, y_pred)))
    print('SVC f1-score  : {:.4f}'.format(f1_score(df_y_test, y_pred)))
    print('SVC precision : {:.4f}'.format(precision_score(df_y_test, y_pred)))
    print('SVC recall    : {:.4f}'.format(recall_score(df_y_test, y_pred)))
    print("\n", classification_report(df_y_test, y_pred))

    cnf_matrix = confusion_matrix(df_y_test, y_pred)
    acc_arr[x] = accuracy_score(df_y_test, y_pred)
    f1_arr[x] = f1_score(df_y_test, y_pred)
    precision_arr[x] = precision_score(df_y_test, y_pred)
    recall_arr[x] = recall_score(df_y_test, y_pred)
    auc_arr[x] = metrics.roc_auc_score(df_y_test, y_prob)
    balanced_accuracy_scores_arr[x] = metrics.balanced_accuracy_score(df_y_test, y_pred)

    tn, fp, fn, tp = confusion_matrix(df_y_test, y_pred).ravel()
    specificity = tn / (tn+fp)
    sensitivity = tp / (tp+fn)
    ppv = tp /(tp+fp)
    npv = tn /(tn+fn)
    pre = tp / (tp + fp)

    sen_arr[x] = sensitivity
    spe_arr[x] = specificity
    ppv_arr[x] = ppv
    npv_arr[x] = npv
    pre_arr[x] = pre




    x = x+ 1


print("%0.2f accuracy with a standard deviation of %0.2f" %
      (100*acc_arr.mean(), 100*acc_arr.std()))
print("%0.2f precision with a standard deviation of %0.2f" %
      (100*precision_arr.mean(), 100*precision_arr.std()))
print("%0.2f auc with a standard deviation of %0.2f" %
      (100*auc_arr.mean(), 100*auc_arr.std()))
print("%0.2f f1 score with a standard deviation of %0.2f" %
      (100*f1_arr.mean(), 100*f1_arr.std()))

print("%0.2f sensitivity with a standard deviation of %0.2f" %
      (100*sen_arr.mean(), 100*sen_arr.std()))

print("%0.2f specificity with a standard deviation of %0.2f" %
      (100*spe_arr.mean(), 100*spe_arr.std()))

print("%0.2f ppv with a standard deviation of %0.2f" %
      (100*ppv_arr.mean(), 100*ppv_arr.std()))
print("%0.2f npv with a standard deviation of %0.2f" %
      (100*npv_arr.mean(), 100*npv_arr.std()))

print("%0.2f recall with a standard deviation of %0.2f" %
      (100*recall_arr.mean(), 100*recall_arr.std()))
print("%0.2f pre with a standard deviation of %0.2f" %
      (100*pre_arr.mean(), 100*pre_arr.std()))
print("%0.2f balanced_accuracy_scores with a standard deviation of %0.2f" %
      (100*balanced_accuracy_scores_arr.mean(), 100*balanced_accuracy_scores_arr.std()))


#### mfcc_default

In [ ]:
import pandas as pd
frame_mfcc = pd.read_pickle('....../COPD/features/20240225_mfccdefault_features.pkl')
frame_mfcc

In [ ]:
# Merge DataFrames on 'name' column
merged_df = pd.merge(frame_mfcc, small_frame[['name']], on='name', how='inner')

# Filter rows in A based on whether 'name' is present in B
frame_mfcc = frame_mfcc[frame_mfcc['name'].isin(merged_df['name'])]


In [ ]:
frame_mfcc = frame_mfcc.reset_index(drop=True)
frame_mfcc

In [ ]:
frame_mfcc['new_feature'] = frame_mfcc['feature'].apply(lambda x: x[:13])
frame_mfcc

In [ ]:
X = frame_mfcc['new_feature'].to_list()
y = frame_mfcc['label'].to_list()

import numpy as np

X = np.vstack(X)
X.shape

In [ ]:
# Assume your DataFrame is df with columns 'new_feature' and 'label'
scaler = StandardScaler()

# Initialize StratifiedKFold
skf = StratifiedKFold(n_splits=10, shuffle=True, random_state=0)

acc_arr = np.empty((10, 1))
f1_arr = np.empty((10, 1))
precision_arr = np.empty((10, 1))
recall_arr = np.empty((10, 1))
auc_arr = np.empty((10, 1))

sen_arr = np.empty((10, 1))
spe_arr = np.empty((10, 1))

ppv_arr = np.empty((10, 1))
npv_arr = np.empty((10, 1))

recall_arr = np.empty((10, 1))
pre_arr = np.empty((10, 1))
balanced_accuracy_scores_arr = np.empty((10, 1))

cnf_arr= []
x = 0

# Loop through each fold
for train_index, test_index in skf.split(X, y):
    X_train, X_test = X[train_index], X[test_index]
    y_train, y_test = np.array(y)[train_index], np.array(y)[test_index]

    # Scale the features using StandardScaler
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)


    df_x_train = X_train_scaled.reshape(-1,  1, X_train_scaled.shape[-1])
    df_x_test = X_test_scaled.reshape(-1,  1, X_test_scaled.shape[-1])
    df_y_train = y_train.reshape(-1, 1)
    df_y_test = y_test.reshape(-1, 1)

    model = load_model('....../0810/3rd_paper_LSTM/features/mfcc_default/{0}'.format(x))


    y_prob = model.predict(df_x_test).ravel()

    pred = model.predict(df_x_test)
    y_pred = [np.round(x) for x in pred]
    test_preds1 = [np.round(x) for x in y_pred]
    test_preds1 = np.array(test_preds1)
    y_pred = test_preds1.flatten('A')

    df_y_test = np.array(df_y_test)
    df_y_test = df_y_test.flatten('A')
    

    print('Accuracy Score: {:.4f}'.format(accuracy_score(df_y_test, y_pred)))
    print('SVC f1-score  : {:.4f}'.format(f1_score(df_y_test, y_pred)))
    print('SVC precision : {:.4f}'.format(precision_score(df_y_test, y_pred)))
    print('SVC recall    : {:.4f}'.format(recall_score(df_y_test, y_pred)))
    print("\n", classification_report(df_y_test, y_pred))

    cnf_matrix = confusion_matrix(df_y_test, y_pred)
    acc_arr[x] = accuracy_score(df_y_test, y_pred)
    f1_arr[x] = f1_score(df_y_test, y_pred)
    precision_arr[x] = precision_score(df_y_test, y_pred)
    recall_arr[x] = recall_score(df_y_test, y_pred)
    auc_arr[x] = metrics.roc_auc_score(df_y_test, y_prob)
    balanced_accuracy_scores_arr[x] = metrics.balanced_accuracy_score(df_y_test, y_pred)

    tn, fp, fn, tp = confusion_matrix(df_y_test, y_pred).ravel()
    specificity = tn / (tn+fp)
    sensitivity = tp / (tp+fn)
    ppv = tp /(tp+fp)
    npv = tn /(tn+fn)
    pre = tp / (tp + fp)

    sen_arr[x] = sensitivity
    spe_arr[x] = specificity
    ppv_arr[x] = ppv
    npv_arr[x] = npv
    pre_arr[x] = pre



    x = x+ 1


print("%0.2f accuracy with a standard deviation of %0.2f" %
      (100*acc_arr.mean(), 100*acc_arr.std()))
print("%0.2f precision with a standard deviation of %0.2f" %
      (100*precision_arr.mean(), 100*precision_arr.std()))
print("%0.2f auc with a standard deviation of %0.2f" %
      (100*auc_arr.mean(), 100*auc_arr.std()))
print("%0.2f f1 score with a standard deviation of %0.2f" %
      (100*f1_arr.mean(), 100*f1_arr.std()))

print("%0.2f sensitivity with a standard deviation of %0.2f" %
      (100*sen_arr.mean(), 100*sen_arr.std()))

print("%0.2f specificity with a standard deviation of %0.2f" %
      (100*spe_arr.mean(), 100*spe_arr.std()))

print("%0.2f ppv with a standard deviation of %0.2f" %
      (100*ppv_arr.mean(), 100*ppv_arr.std()))
print("%0.2f npv with a standard deviation of %0.2f" %
      (100*npv_arr.mean(), 100*npv_arr.std()))

print("%0.2f recall with a standard deviation of %0.2f" %
      (100*recall_arr.mean(), 100*recall_arr.std()))
print("%0.2f pre with a standard deviation of %0.2f" %
      (100*pre_arr.mean(), 100*pre_arr.std()))
print("%0.2f balanced_accuracy_scores with a standard deviation of %0.2f" %
      (100*balanced_accuracy_scores_arr.mean(), 100*balanced_accuracy_scores_arr.std()))


#### LPCC

In [ ]:
import pandas as pd
frame_mfcc = pd.read_pickle('....../COPD/features/20240225_lpcc_features.pkl')
frame_mfcc

In [ ]:
# Merge DataFrames on 'name' column
merged_df = pd.merge(frame_mfcc, small_frame[['name']], on='name', how='inner')

# Filter rows in A based on whether 'name' is present in B
frame_mfcc = frame_mfcc[frame_mfcc['name'].isin(merged_df['name'])]

In [ ]:
frame_mfcc = frame_mfcc.reset_index(drop=True)
frame_mfcc

In [ ]:
frame_mfcc['new_feature'] = frame_mfcc['feature'].apply(lambda x: x[:13])
frame_mfcc

In [ ]:
X = frame_mfcc['new_feature'].to_list()
y = frame_mfcc['label'].to_list()

import numpy as np

X = np.vstack(X)
X.shape

In [ ]:
# Assume your DataFrame is df with columns 'new_feature' and 'label'
scaler = StandardScaler()

# Initialize StratifiedKFold
skf = StratifiedKFold(n_splits=10, shuffle=True, random_state=0)

acc_arr = np.empty((10, 1))
f1_arr = np.empty((10, 1))
precision_arr = np.empty((10, 1))
recall_arr = np.empty((10, 1))
auc_arr = np.empty((10, 1))

sen_arr = np.empty((10, 1))
spe_arr = np.empty((10, 1))

ppv_arr = np.empty((10, 1))
npv_arr = np.empty((10, 1))

recall_arr = np.empty((10, 1))
pre_arr = np.empty((10, 1))
balanced_accuracy_scores_arr = np.empty((10, 1))

cnf_arr= []
x = 0

# Loop through each fold
for train_index, test_index in skf.split(X, y):
    X_train, X_test = X[train_index], X[test_index]
    y_train, y_test = np.array(y)[train_index], np.array(y)[test_index]

    # Scale the features using StandardScaler
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)


    df_x_train = X_train_scaled.reshape(-1,  1, X_train_scaled.shape[-1])
    df_x_test = X_test_scaled.reshape(-1,  1, X_test_scaled.shape[-1])
    df_y_train = y_train.reshape(-1, 1)
    df_y_test = y_test.reshape(-1, 1)

    model = load_model('....../0810/3rd_paper_LSTM/features/lpcc/{0}'.format(x))


    y_prob = model.predict(df_x_test).ravel()

    pred = model.predict(df_x_test)
    y_pred = [np.round(x) for x in pred]
    test_preds1 = [np.round(x) for x in y_pred]
    test_preds1 = np.array(test_preds1)
    y_pred = test_preds1.flatten('A')

    df_y_test = np.array(df_y_test)
    df_y_test = df_y_test.flatten('A')
    

    print('Accuracy Score: {:.4f}'.format(accuracy_score(df_y_test, y_pred)))
    print('SVC f1-score  : {:.4f}'.format(f1_score(df_y_test, y_pred)))
    print('SVC precision : {:.4f}'.format(precision_score(df_y_test, y_pred)))
    print('SVC recall    : {:.4f}'.format(recall_score(df_y_test, y_pred)))
    print("\n", classification_report(df_y_test, y_pred))

    cnf_matrix = confusion_matrix(df_y_test, y_pred)
    acc_arr[x] = accuracy_score(df_y_test, y_pred)
    f1_arr[x] = f1_score(df_y_test, y_pred)
    precision_arr[x] = precision_score(df_y_test, y_pred)
    recall_arr[x] = recall_score(df_y_test, y_pred)
    auc_arr[x] = metrics.roc_auc_score(df_y_test, y_prob)
    balanced_accuracy_scores_arr[x] = metrics.balanced_accuracy_score(df_y_test, y_pred)

    tn, fp, fn, tp = confusion_matrix(df_y_test, y_pred).ravel()
    specificity = tn / (tn+fp)
    sensitivity = tp / (tp+fn)
    ppv = tp /(tp+fp)
    npv = tn /(tn+fn)
    pre = tp / (tp + fp)

    sen_arr[x] = sensitivity
    spe_arr[x] = specificity
    ppv_arr[x] = ppv
    npv_arr[x] = npv
    pre_arr[x] = pre



    x = x+ 1


print("%0.2f accuracy with a standard deviation of %0.2f" %
      (100*acc_arr.mean(), 100*acc_arr.std()))
print("%0.2f precision with a standard deviation of %0.2f" %
      (100*precision_arr.mean(), 100*precision_arr.std()))
print("%0.2f auc with a standard deviation of %0.2f" %
      (100*auc_arr.mean(), 100*auc_arr.std()))
print("%0.2f f1 score with a standard deviation of %0.2f" %
      (100*f1_arr.mean(), 100*f1_arr.std()))

print("%0.2f sensitivity with a standard deviation of %0.2f" %
      (100*sen_arr.mean(), 100*sen_arr.std()))

print("%0.2f specificity with a standard deviation of %0.2f" %
      (100*spe_arr.mean(), 100*spe_arr.std()))

print("%0.2f ppv with a standard deviation of %0.2f" %
      (100*ppv_arr.mean(), 100*ppv_arr.std()))
print("%0.2f npv with a standard deviation of %0.2f" %
      (100*npv_arr.mean(), 100*npv_arr.std()))

print("%0.2f recall with a standard deviation of %0.2f" %
      (100*recall_arr.mean(), 100*recall_arr.std()))
print("%0.2f pre with a standard deviation of %0.2f" %
      (100*pre_arr.mean(), 100*pre_arr.std()))
print("%0.2f balanced_accuracy_scores with a standard deviation of %0.2f" %
      (100*balanced_accuracy_scores_arr.mean(), 100*balanced_accuracy_scores_arr.std()))

#### eGeMAPS

In [ ]:
import pandas as pd
frame_mfcc = pd.read_pickle('....../COPD/features/20240129_5460_eGeMAPSv01b_88.pkl')
frame_mfcc

In [ ]:
# Merge DataFrames on 'name' column
merged_df = pd.merge(frame_mfcc, small_frame[['name']], on='name', how='inner')

# Filter rows in A based on whether 'name' is present in B
frame_mfcc = frame_mfcc[frame_mfcc['name'].isin(merged_df['name'])]

In [ ]:
frame_mfcc = frame_mfcc.reset_index(drop=True)
frame_mfcc

In [ ]:
X = frame_mfcc.iloc[ :, :88]
y = frame_mfcc['label'].to_list()



In [ ]:
# Assume your DataFrame is df with columns 'new_feature' and 'label'
scaler = StandardScaler()

# Initialize StratifiedKFold
skf = StratifiedKFold(n_splits=10, shuffle=True, random_state=0)

acc_arr = np.empty((10, 1))
f1_arr = np.empty((10, 1))
precision_arr = np.empty((10, 1))
recall_arr = np.empty((10, 1))
auc_arr = np.empty((10, 1))

sen_arr = np.empty((10, 1))
spe_arr = np.empty((10, 1))

ppv_arr = np.empty((10, 1))
npv_arr = np.empty((10, 1))

recall_arr = np.empty((10, 1))
pre_arr = np.empty((10, 1))
balanced_accuracy_scores_arr = np.empty((10, 1))

cnf_arr= []
x = 0

# Loop through each fold
for train_index, test_index in skf.split(X, y):
    X_train, X_test = X.iloc[train_index], X.iloc[test_index]
    y_train, y_test = np.array(y)[train_index], np.array(y)[test_index]

    # Scale the features using StandardScaler
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)


    df_x_train = X_train_scaled.reshape(-1,  1, X_train_scaled.shape[-1])
    df_x_test = X_test_scaled.reshape(-1,  1, X_test_scaled.shape[-1])
    df_y_train = y_train.reshape(-1, 1)
    df_y_test = y_test.reshape(-1, 1)

    model = load_model('....../0810/3rd_paper_LSTM/features/egemaps/{0}'.format(x))


    y_prob = model.predict(df_x_test).ravel()

    pred = model.predict(df_x_test)
    y_pred = [np.round(x) for x in pred]
    test_preds1 = [np.round(x) for x in y_pred]
    test_preds1 = np.array(test_preds1)
    y_pred = test_preds1.flatten('A')

    df_y_test = np.array(df_y_test)
    df_y_test = df_y_test.flatten('A')
    

    print('Accuracy Score: {:.4f}'.format(accuracy_score(df_y_test, y_pred)))
    print('SVC f1-score  : {:.4f}'.format(f1_score(df_y_test, y_pred)))
    print('SVC precision : {:.4f}'.format(precision_score(df_y_test, y_pred)))
    print('SVC recall    : {:.4f}'.format(recall_score(df_y_test, y_pred)))
    print("\n", classification_report(df_y_test, y_pred))

    cnf_matrix = confusion_matrix(df_y_test, y_pred)
    acc_arr[x] = accuracy_score(df_y_test, y_pred)
    f1_arr[x] = f1_score(df_y_test, y_pred)
    precision_arr[x] = precision_score(df_y_test, y_pred)
    recall_arr[x] = recall_score(df_y_test, y_pred)
    auc_arr[x] = metrics.roc_auc_score(df_y_test, y_prob)
    balanced_accuracy_scores_arr[x] = metrics.balanced_accuracy_score(df_y_test, y_pred)

    tn, fp, fn, tp = confusion_matrix(df_y_test, y_pred).ravel()
    specificity = tn / (tn+fp)
    sensitivity = tp / (tp+fn)
    ppv = tp /(tp+fp)
    npv = tn /(tn+fn)
    pre = tp / (tp + fp)

    sen_arr[x] = sensitivity
    spe_arr[x] = specificity
    ppv_arr[x] = ppv
    npv_arr[x] = npv
    pre_arr[x] = pre



    x = x+ 1


print("%0.2f accuracy with a standard deviation of %0.2f" %
      (100*acc_arr.mean(), 100*acc_arr.std()))
print("%0.2f precision with a standard deviation of %0.2f" %
      (100*precision_arr.mean(), 100*precision_arr.std()))
print("%0.2f auc with a standard deviation of %0.2f" %
      (100*auc_arr.mean(), 100*auc_arr.std()))
print("%0.2f f1 score with a standard deviation of %0.2f" %
      (100*f1_arr.mean(), 100*f1_arr.std()))

print("%0.2f sensitivity with a standard deviation of %0.2f" %
      (100*sen_arr.mean(), 100*sen_arr.std()))

print("%0.2f specificity with a standard deviation of %0.2f" %
      (100*spe_arr.mean(), 100*spe_arr.std()))

print("%0.2f ppv with a standard deviation of %0.2f" %
      (100*ppv_arr.mean(), 100*ppv_arr.std()))
print("%0.2f npv with a standard deviation of %0.2f" %
      (100*npv_arr.mean(), 100*npv_arr.std()))

print("%0.2f recall with a standard deviation of %0.2f" %
      (100*recall_arr.mean(), 100*recall_arr.std()))
print("%0.2f pre with a standard deviation of %0.2f" %
      (100*pre_arr.mean(), 100*pre_arr.std()))
print("%0.2f balanced_accuracy_scores with a standard deviation of %0.2f" %
      (100*balanced_accuracy_scores_arr.mean(), 100*balanced_accuracy_scores_arr.std()))

#### phonation_features

In [ ]:
import pandas as pd
frame_mfcc = pd.read_pickle('....../COPD/features/20240225_phonation_features_Disvoice.pkl')
frame_mfcc

In [ ]:
frame_mfcc = frame_mfcc.dropna()
frame_mfcc

In [ ]:
X = frame_mfcc.iloc[ :, :28]
y = frame_mfcc['label'].to_list()

In [ ]:
# Assume your DataFrame is df with columns 'new_feature' and 'label'
scaler = StandardScaler()

# Initialize StratifiedKFold
skf = StratifiedKFold(n_splits=10, shuffle=True, random_state=0)

acc_arr = np.empty((10, 1))
f1_arr = np.empty((10, 1))
precision_arr = np.empty((10, 1))
recall_arr = np.empty((10, 1))
auc_arr = np.empty((10, 1))

sen_arr = np.empty((10, 1))
spe_arr = np.empty((10, 1))

ppv_arr = np.empty((10, 1))
npv_arr = np.empty((10, 1))

recall_arr = np.empty((10, 1))
pre_arr = np.empty((10, 1))
balanced_accuracy_scores_arr = np.empty((10, 1))

cnf_arr= []
x = 0

# Loop through each fold
for train_index, test_index in skf.split(X, y):
    X_train, X_test = X.iloc[train_index], X.iloc[test_index]
    y_train, y_test = np.array(y)[train_index], np.array(y)[test_index]

    # Scale the features using StandardScaler
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)


    df_x_train = X_train_scaled.reshape(-1,  1, X_train_scaled.shape[-1])
    df_x_test = X_test_scaled.reshape(-1,  1, X_test_scaled.shape[-1])
    df_y_train = y_train.reshape(-1, 1)
    df_y_test = y_test.reshape(-1, 1)

    model = load_model('....../0810/3rd_paper_LSTM/features/phonation/{0}'.format(x))


    y_prob = model.predict(df_x_test).ravel()

    pred = model.predict(df_x_test)
    y_pred = [np.round(x) for x in pred]
    test_preds1 = [np.round(x) for x in y_pred]
    test_preds1 = np.array(test_preds1)
    y_pred = test_preds1.flatten('A')

    df_y_test = np.array(df_y_test)
    df_y_test = df_y_test.flatten('A')
    

    print('Accuracy Score: {:.4f}'.format(accuracy_score(df_y_test, y_pred)))
    print('SVC f1-score  : {:.4f}'.format(f1_score(df_y_test, y_pred)))
    print('SVC precision : {:.4f}'.format(precision_score(df_y_test, y_pred)))
    print('SVC recall    : {:.4f}'.format(recall_score(df_y_test, y_pred)))
    print("\n", classification_report(df_y_test, y_pred))

    cnf_matrix = confusion_matrix(df_y_test, y_pred)
    acc_arr[x] = accuracy_score(df_y_test, y_pred)
    f1_arr[x] = f1_score(df_y_test, y_pred)
    precision_arr[x] = precision_score(df_y_test, y_pred)
    recall_arr[x] = recall_score(df_y_test, y_pred)
    auc_arr[x] = metrics.roc_auc_score(df_y_test, y_prob)
    balanced_accuracy_scores_arr[x] = metrics.balanced_accuracy_score(df_y_test, y_pred)

    tn, fp, fn, tp = confusion_matrix(df_y_test, y_pred).ravel()
    specificity = tn / (tn+fp)
    sensitivity = tp / (tp+fn)
    ppv = tp /(tp+fp)
    npv = tn /(tn+fn)
    pre = tp / (tp + fp)

    sen_arr[x] = sensitivity
    spe_arr[x] = specificity
    ppv_arr[x] = ppv
    npv_arr[x] = npv
    pre_arr[x] = pre



    x = x+ 1


print("%0.2f accuracy with a standard deviation of %0.2f" %
      (100*acc_arr.mean(), 100*acc_arr.std()))
print("%0.2f precision with a standard deviation of %0.2f" %
      (100*precision_arr.mean(), 100*precision_arr.std()))
print("%0.2f auc with a standard deviation of %0.2f" %
      (100*auc_arr.mean(), 100*auc_arr.std()))
print("%0.2f f1 score with a standard deviation of %0.2f" %
      (100*f1_arr.mean(), 100*f1_arr.std()))

print("%0.2f sensitivity with a standard deviation of %0.2f" %
      (100*sen_arr.mean(), 100*sen_arr.std()))

print("%0.2f specificity with a standard deviation of %0.2f" %
      (100*spe_arr.mean(), 100*spe_arr.std()))

print("%0.2f ppv with a standard deviation of %0.2f" %
      (100*ppv_arr.mean(), 100*ppv_arr.std()))
print("%0.2f npv with a standard deviation of %0.2f" %
      (100*npv_arr.mean(), 100*npv_arr.std()))

print("%0.2f recall with a standard deviation of %0.2f" %
      (100*recall_arr.mean(), 100*recall_arr.std()))
print("%0.2f pre with a standard deviation of %0.2f" %
      (100*pre_arr.mean(), 100*pre_arr.std()))
print("%0.2f balanced_accuracy_scores with a standard deviation of %0.2f" %
      (100*balanced_accuracy_scores_arr.mean(), 100*balanced_accuracy_scores_arr.std()))

#### RepLearning_features

In [ ]:
import pandas as pd
frame_mfcc = pd.read_pickle('....../COPD/features/20240225_RepLearning_features_Disvoice.pkl')
frame_mfcc

In [ ]:
# Merge DataFrames on 'name' column
merged_df = pd.merge(frame_mfcc, small_frame[['name']], on='name', how='inner')

# Filter rows in A based on whether 'name' is present in B
frame_mfcc = frame_mfcc[frame_mfcc['name'].isin(merged_df['name'])]

In [ ]:
frame_mfcc = frame_mfcc.reset_index(drop=True)
frame_mfcc

In [ ]:
frame_mfcc = frame_mfcc.dropna()
frame_mfcc

In [ ]:
X = frame_mfcc.iloc[ :, :1536]
y = frame_mfcc['label'].to_list()

In [ ]:
# Assume your DataFrame is df with columns 'new_feature' and 'label'
scaler = StandardScaler()

# Initialize StratifiedKFold
skf = StratifiedKFold(n_splits=10, shuffle=True, random_state=0)

acc_arr = np.empty((10, 1))
f1_arr = np.empty((10, 1))
precision_arr = np.empty((10, 1))
recall_arr = np.empty((10, 1))
auc_arr = np.empty((10, 1))

sen_arr = np.empty((10, 1))
spe_arr = np.empty((10, 1))

ppv_arr = np.empty((10, 1))
npv_arr = np.empty((10, 1))

recall_arr = np.empty((10, 1))
pre_arr = np.empty((10, 1))
balanced_accuracy_scores_arr = np.empty((10, 1))

cnf_arr= []
x = 0

# Loop through each fold
for train_index, test_index in skf.split(X, y):
    X_train, X_test = X.iloc[train_index], X.iloc[test_index]
    y_train, y_test = np.array(y)[train_index], np.array(y)[test_index]

    # Scale the features using StandardScaler
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)


    df_x_train = X_train_scaled.reshape(-1,  1, X_train_scaled.shape[-1])
    df_x_test = X_test_scaled.reshape(-1,  1, X_test_scaled.shape[-1])
    df_y_train = y_train.reshape(-1, 1)
    df_y_test = y_test.reshape(-1, 1)

    model = load_model('....../0810/3rd_paper_LSTM/features/replearning94.14/{0}'.format(x))


    y_prob = model.predict(df_x_test).ravel()

    pred = model.predict(df_x_test)
    y_pred = [np.round(x) for x in pred]
    test_preds1 = [np.round(x) for x in y_pred]
    test_preds1 = np.array(test_preds1)
    y_pred = test_preds1.flatten('A')

    df_y_test = np.array(df_y_test)
    df_y_test = df_y_test.flatten('A')
    

    print('Accuracy Score: {:.4f}'.format(accuracy_score(df_y_test, y_pred)))
    print('SVC f1-score  : {:.4f}'.format(f1_score(df_y_test, y_pred)))
    print('SVC precision : {:.4f}'.format(precision_score(df_y_test, y_pred)))
    print('SVC recall    : {:.4f}'.format(recall_score(df_y_test, y_pred)))
    print("\n", classification_report(df_y_test, y_pred))

    cnf_matrix = confusion_matrix(df_y_test, y_pred)
    acc_arr[x] = accuracy_score(df_y_test, y_pred)
    f1_arr[x] = f1_score(df_y_test, y_pred)
    precision_arr[x] = precision_score(df_y_test, y_pred)
    recall_arr[x] = recall_score(df_y_test, y_pred)
    auc_arr[x] = metrics.roc_auc_score(df_y_test, y_prob)
    balanced_accuracy_scores_arr[x] = metrics.balanced_accuracy_score(df_y_test, y_pred)

    tn, fp, fn, tp = confusion_matrix(df_y_test, y_pred).ravel()
    specificity = tn / (tn+fp)
    sensitivity = tp / (tp+fn)
    ppv = tp /(tp+fp)
    npv = tn /(tn+fn)
    pre = tp / (tp + fp)

    sen_arr[x] = sensitivity
    spe_arr[x] = specificity
    ppv_arr[x] = ppv
    npv_arr[x] = npv
    pre_arr[x] = pre


    x = x+ 1


print("%0.2f accuracy with a standard deviation of %0.2f" %
      (100*acc_arr.mean(), 100*acc_arr.std()))
print("%0.2f precision with a standard deviation of %0.2f" %
      (100*precision_arr.mean(), 100*precision_arr.std()))
print("%0.2f auc with a standard deviation of %0.2f" %
      (100*auc_arr.mean(), 100*auc_arr.std()))
print("%0.2f f1 score with a standard deviation of %0.2f" %
      (100*f1_arr.mean(), 100*f1_arr.std()))

print("%0.2f sensitivity with a standard deviation of %0.2f" %
      (100*sen_arr.mean(), 100*sen_arr.std()))

print("%0.2f specificity with a standard deviation of %0.2f" %
      (100*spe_arr.mean(), 100*spe_arr.std()))

print("%0.2f ppv with a standard deviation of %0.2f" %
      (100*ppv_arr.mean(), 100*ppv_arr.std()))
print("%0.2f npv with a standard deviation of %0.2f" %
      (100*npv_arr.mean(), 100*npv_arr.std()))

print("%0.2f recall with a standard deviation of %0.2f" %
      (100*recall_arr.mean(), 100*recall_arr.std()))
print("%0.2f pre with a standard deviation of %0.2f" %
      (100*pre_arr.mean(), 100*pre_arr.std()))
print("%0.2f balanced_accuracy_scores with a standard deviation of %0.2f" %
      (100*balanced_accuracy_scores_arr.mean(), 100*balanced_accuracy_scores_arr.std()))

#### VGGish

In [ ]:
import pandas as pd
frame_mfcc = pd.read_pickle('....../COPD/features/20240225_vggish_features.pkl')
frame_mfcc

In [ ]:
# Merge DataFrames on 'name' column
merged_df = pd.merge(frame_mfcc, small_frame[['name']], on='name', how='inner')

# Filter rows in A based on whether 'name' is present in B
frame_mfcc = frame_mfcc[frame_mfcc['name'].isin(merged_df['name'])]

In [ ]:
frame_mfcc = frame_mfcc.reset_index(drop=True)
frame_mfcc

In [ ]:
import numpy as np


frame_mfcc = frame_mfcc[frame_mfcc['feature'].apply(lambda x: not any(np.isnan(x)))]
frame_mfcc

In [ ]:
frame_mfcc['new_feature'] = frame_mfcc['feature'].apply(lambda x: x[:128])
frame_mfcc

In [ ]:
X = frame_mfcc['new_feature'].to_list()
y = frame_mfcc['label'].to_list()

import numpy as np

X = np.vstack(X)
X.shape


In [ ]:
# Assume your DataFrame is df with columns 'new_feature' and 'label'
scaler = StandardScaler()

# Initialize StratifiedKFold
skf = StratifiedKFold(n_splits=10, shuffle=True, random_state=0)

acc_arr = np.empty((10, 1))
f1_arr = np.empty((10, 1))
precision_arr = np.empty((10, 1))
recall_arr = np.empty((10, 1))
auc_arr = np.empty((10, 1))

sen_arr = np.empty((10, 1))
spe_arr = np.empty((10, 1))

ppv_arr = np.empty((10, 1))
npv_arr = np.empty((10, 1))

recall_arr = np.empty((10, 1))
pre_arr = np.empty((10, 1))
balanced_accuracy_scores_arr = np.empty((10, 1))

cnf_arr= []
x = 0

# Loop through each fold
for train_index, test_index in skf.split(X, y):
    X_train, X_test = X[train_index], X[test_index]
    y_train, y_test = np.array(y)[train_index], np.array(y)[test_index]

    # Scale the features using StandardScaler
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)


    df_x_train = X_train_scaled.reshape(-1,  1, X_train_scaled.shape[-1])
    df_x_test = X_test_scaled.reshape(-1,  1, X_test_scaled.shape[-1])
    df_y_train = y_train.reshape(-1, 1)
    df_y_test = y_test.reshape(-1, 1)

    model = load_model('....../0810/3rd_paper_LSTM/features/vggish/{0}'.format(x))


    y_prob = model.predict(df_x_test).ravel()

    pred = model.predict(df_x_test)
    y_pred = [np.round(x) for x in pred]
    test_preds1 = [np.round(x) for x in y_pred]
    test_preds1 = np.array(test_preds1)
    y_pred = test_preds1.flatten('A')

    df_y_test = np.array(df_y_test)
    df_y_test = df_y_test.flatten('A')
    

    print('Accuracy Score: {:.4f}'.format(accuracy_score(df_y_test, y_pred)))
    print('SVC f1-score  : {:.4f}'.format(f1_score(df_y_test, y_pred)))
    print('SVC precision : {:.4f}'.format(precision_score(df_y_test, y_pred)))
    print('SVC recall    : {:.4f}'.format(recall_score(df_y_test, y_pred)))
    print("\n", classification_report(df_y_test, y_pred))

    cnf_matrix = confusion_matrix(df_y_test, y_pred)
    acc_arr[x] = accuracy_score(df_y_test, y_pred)
    f1_arr[x] = f1_score(df_y_test, y_pred)
    precision_arr[x] = precision_score(df_y_test, y_pred)
    recall_arr[x] = recall_score(df_y_test, y_pred)
    auc_arr[x] = metrics.roc_auc_score(df_y_test, y_prob)
    balanced_accuracy_scores_arr[x] = metrics.balanced_accuracy_score(df_y_test, y_pred)

    tn, fp, fn, tp = confusion_matrix(df_y_test, y_pred).ravel()
    specificity = tn / (tn+fp)
    sensitivity = tp / (tp+fn)
    ppv = tp /(tp+fp)
    npv = tn /(tn+fn)
    pre = tp / (tp + fp)

    sen_arr[x] = sensitivity
    spe_arr[x] = specificity
    ppv_arr[x] = ppv
    npv_arr[x] = npv
    pre_arr[x] = pre



    x = x+ 1


print("%0.2f accuracy with a standard deviation of %0.2f" %
      (100*acc_arr.mean(), 100*acc_arr.std()))
print("%0.2f precision with a standard deviation of %0.2f" %
      (100*precision_arr.mean(), 100*precision_arr.std()))
print("%0.2f auc with a standard deviation of %0.2f" %
      (100*auc_arr.mean(), 100*auc_arr.std()))
print("%0.2f f1 score with a standard deviation of %0.2f" %
      (100*f1_arr.mean(), 100*f1_arr.std()))

print("%0.2f sensitivity with a standard deviation of %0.2f" %
      (100*sen_arr.mean(), 100*sen_arr.std()))

print("%0.2f specificity with a standard deviation of %0.2f" %
      (100*spe_arr.mean(), 100*spe_arr.std()))

print("%0.2f ppv with a standard deviation of %0.2f" %
      (100*ppv_arr.mean(), 100*ppv_arr.std()))
print("%0.2f npv with a standard deviation of %0.2f" %
      (100*npv_arr.mean(), 100*npv_arr.std()))

print("%0.2f recall with a standard deviation of %0.2f" %
      (100*recall_arr.mean(), 100*recall_arr.std()))
print("%0.2f pre with a standard deviation of %0.2f" %
      (100*pre_arr.mean(), 100*pre_arr.std()))
print("%0.2f balanced_accuracy_scores with a standard deviation of %0.2f" %
      (100*balanced_accuracy_scores_arr.mean(), 100*balanced_accuracy_scores_arr.std()))

In [ ]:
#### mfcc+eGeMAPS

#### mfcc+eGeMAPS

In [ ]:
import pandas as pd
frame_mfcc = pd.read_pickle('....../COPD/features/20240225_mfcc_features.pkl')
frame_mfcc

In [ ]:
frame_mfcc['new_feature'] = frame_mfcc['feature'].apply(lambda x: x[:30])
frame_mfcc

In [ ]:
import pandas as pd
frame_GeMAP = pd.read_pickle('....../COPD/features/20240129_5460_eGeMAPSv01b_88.pkl')
frame_GeMAP

In [ ]:
frame_GeMAP_Energy_Amplitude  = frame_GeMAP[[

                                               'F0semitoneFrom27.5Hz_sma3nz_amean','F0semitoneFrom27.5Hz_sma3nz_stddevNorm',
                                             'jitterLocal_sma3nz_amean', 'jitterLocal_sma3nz_stddevNorm', 
                                              'F1frequency_sma3nz_amean', 'F1frequency_sma3nz_stddevNorm',  
                                              'F2frequency_sma3nz_amean',  'F2frequency_sma3nz_stddevNorm', 
                                               'F3frequency_sma3nz_stddevNorm','F3amplitudeLogRelF0_sma3nz_amean',
                                               'F1bandwidth_sma3nz_amean', 'F1bandwidth_sma3nz_stddevNorm',

                                             


                                             'label','name'
                                             
                                             ]]

frame_GeMAP_Energy_Amplitude    

In [ ]:
frame_GeMAP_Energy_Amplitude  = frame_GeMAP[['shimmerLocaldB_sma3nz_amean', 'shimmerLocaldB_sma3nz_stddevNorm',
                                             'loudness_sma3_amean', 'loudness_sma3_stddevNorm', 
                                             'HNRdBACF_sma3nz_amean', 'HNRdBACF_sma3nz_stddevNorm',

                                             'loudnessPeaksPerSec',
                                             'MeanVoicedSegmentLengthSec', 'StddevVoicedSegmentLengthSec',
                                              'MeanUnvoicedSegmentLength', 'StddevUnvoicedSegmentLength',
                                               'VoicedSegmentsPerSec',

                                               'F0semitoneFrom27.5Hz_sma3nz_amean','F0semitoneFrom27.5Hz_sma3nz_stddevNorm',
                                             'jitterLocal_sma3nz_amean', 'jitterLocal_sma3nz_stddevNorm', 
                                              'F1frequency_sma3nz_amean', 'F1frequency_sma3nz_stddevNorm',  
                                              'F2frequency_sma3nz_amean',  'F2frequency_sma3nz_stddevNorm', 
                                               'F3frequency_sma3nz_stddevNorm','F3amplitudeLogRelF0_sma3nz_amean',
                                               'F1bandwidth_sma3nz_amean', 'F1bandwidth_sma3nz_stddevNorm',


                                               
                                               

                                             

                                             


                                             'label','name'
                                             
                                             ]]

frame_GeMAP_Energy_Amplitude    

In [ ]:
frame_GeMAP_Energy_Amplitude  = frame_GeMAP[['shimmerLocaldB_sma3nz_amean', 'shimmerLocaldB_sma3nz_stddevNorm',
                                             'loudness_sma3_amean', 'loudness_sma3_stddevNorm', 
                                             'HNRdBACF_sma3nz_amean', 'HNRdBACF_sma3nz_stddevNorm',


                                              'loudnessPeaksPerSec',
                                             'MeanVoicedSegmentLengthSec', 'StddevVoicedSegmentLengthSec',
                                              'MeanUnvoicedSegmentLength', 'StddevUnvoicedSegmentLength',
                                               'VoicedSegmentsPerSec',


                                             'label','name'
                                             
                                             ]]

frame_GeMAP_Energy_Amplitude    

In [ ]:
frame_GeMAP_Energy_Amplitude  = frame_GeMAP[[
    
    'shimmerLocaldB_sma3nz_amean', 'shimmerLocaldB_sma3nz_stddevNorm',
                                             'loudness_sma3_amean', 'loudness_sma3_stddevNorm', 
                                             'HNRdBACF_sma3nz_amean', 'HNRdBACF_sma3nz_stddevNorm',
                                             


                                             'label','name'
                                             
                                             ]]

frame_GeMAP_Energy_Amplitude    

In [ ]:
list40 = []
for i in range(0,30):
        list40.append('mfcc'+str(i))
frame_mfcc1 = pd.DataFrame(frame_mfcc['new_feature'].to_list(), columns= list40)
frame_mfcc1

In [ ]:
result_df = pd.concat([frame_mfcc1, frame_GeMAP_Energy_Amplitude  ], axis=1)
result_df

In [ ]:
# Merge DataFrames on 'name' column
merged_df = pd.merge(result_df, small_frame[['name']], on='name', how='inner')


# Filter rows in A based on whether 'name' is present in B
frame_result_df = result_df[result_df['name'].isin(merged_df['name'])]
frame_result_df

In [ ]:
X = frame_result_df.iloc[ :, :54]
y = frame_result_df['label'].to_list()


In [ ]:
import numpy as np
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, roc_auc_score
from sklearn.metrics import confusion_matrix
from sklearn import metrics
from sklearn.metrics import classification_report
from keras.models import load_model
from keras.models import save_model


# Assume your DataFrame is df with columns 'new_feature' and 'label'
scaler = StandardScaler()

# Initialize StratifiedKFold
skf = StratifiedKFold(n_splits=10, shuffle=True, random_state=0)

acc_arr = np.empty((10, 1))
f1_arr = np.empty((10, 1))
precision_arr = np.empty((10, 1))
recall_arr = np.empty((10, 1))
auc_arr = np.empty((10, 1))

sen_arr = np.empty((10, 1))
spe_arr = np.empty((10, 1))

ppv_arr = np.empty((10, 1))
npv_arr = np.empty((10, 1))

recall_arr = np.empty((10, 1))
pre_arr = np.empty((10, 1))
balanced_accuracy_scores_arr = np.empty((10, 1))

cnf_arr= []
x = 0

# Loop through each fold
for train_index, test_index in skf.split(X, y):
    X_train, X_test = X.iloc[train_index], X.iloc[test_index]
    y_train, y_test = np.array(y)[train_index], np.array(y)[test_index]

    # Scale the features using StandardScaler
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)


    df_x_train = X_train_scaled.reshape(-1,  1, X_train_scaled.shape[-1])
    df_x_test = X_test_scaled.reshape(-1,  1, X_test_scaled.shape[-1])
    df_y_train = y_train.reshape(-1, 1)
    df_y_test = y_test.reshape(-1, 1)

    model = load_model('....../0810/3rd_paper_LSTM/domains_fusion/mfcc_all/{0}'.format(x))


    y_prob = model.predict(df_x_test).ravel()

    pred = model.predict(df_x_test)
    y_pred = [np.round(x) for x in pred]
    test_preds1 = [np.round(x) for x in y_pred]
    test_preds1 = np.array(test_preds1)
    y_pred = test_preds1.flatten('A')

    df_y_test = np.array(df_y_test)
    df_y_test = df_y_test.flatten('A')
    

    print('Accuracy Score: {:.4f}'.format(accuracy_score(df_y_test, y_pred)))
    print('SVC f1-score  : {:.4f}'.format(f1_score(df_y_test, y_pred)))
    print('SVC precision : {:.4f}'.format(precision_score(df_y_test, y_pred)))
    print('SVC recall    : {:.4f}'.format(recall_score(df_y_test, y_pred)))
    print("\n", classification_report(df_y_test, y_pred))

    cnf_matrix = confusion_matrix(df_y_test, y_pred)
    acc_arr[x] = accuracy_score(df_y_test, y_pred)
    f1_arr[x] = f1_score(df_y_test, y_pred)
    precision_arr[x] = precision_score(df_y_test, y_pred)
    recall_arr[x] = recall_score(df_y_test, y_pred)
    auc_arr[x] = metrics.roc_auc_score(df_y_test, y_prob)
    balanced_accuracy_scores_arr[x] = metrics.balanced_accuracy_score(df_y_test, y_pred)

    tn, fp, fn, tp = confusion_matrix(df_y_test, y_pred).ravel()
    specificity = tn / (tn+fp)
    sensitivity = tp / (tp+fn)
    ppv = tp /(tp+fp)
    npv = tn /(tn+fn)
    pre = tp / (tp + fp)

    sen_arr[x] = sensitivity
    spe_arr[x] = specificity
    ppv_arr[x] = ppv
    npv_arr[x] = npv
    pre_arr[x] = pre



    x = x+ 1


print("%0.2f accuracy with a standard deviation of %0.2f" %
      (100*acc_arr.mean(), 100*acc_arr.std()))
print("%0.2f precision with a standard deviation of %0.2f" %
      (100*precision_arr.mean(), 100*precision_arr.std()))
print("%0.2f auc with a standard deviation of %0.2f" %
      (100*auc_arr.mean(), 100*auc_arr.std()))
print("%0.2f f1 score with a standard deviation of %0.2f" %
      (100*f1_arr.mean(), 100*f1_arr.std()))

print("%0.2f sensitivity with a standard deviation of %0.2f" %
      (100*sen_arr.mean(), 100*sen_arr.std()))

print("%0.2f specificity with a standard deviation of %0.2f" %
      (100*spe_arr.mean(), 100*spe_arr.std()))

print("%0.2f ppv with a standard deviation of %0.2f" %
      (100*ppv_arr.mean(), 100*ppv_arr.std()))
print("%0.2f npv with a standard deviation of %0.2f" %
      (100*npv_arr.mean(), 100*npv_arr.std()))

print("%0.2f recall with a standard deviation of %0.2f" %
      (100*recall_arr.mean(), 100*recall_arr.std()))
print("%0.2f pre with a standard deviation of %0.2f" %
      (100*pre_arr.mean(), 100*pre_arr.std()))
print("%0.2f balanced_accuracy_scores with a standard deviation of %0.2f" %
      (100*balanced_accuracy_scores_arr.mean(), 100*balanced_accuracy_scores_arr.std()))

In [ ]:
auc_arr